# CXR Essential-Tag Evaluator (series-level, IOD-free)

X-ray **essential tags** 기준으로 DICOM 메타데이터의 품질을 **series 단위**로 평가한다.

- 기존 `DicomCodeStandardEvaluator` 와 달리 **IOD / Type 을 고려하지 않고**,
  essential tag 목록에 있는 모든 태그를 **Type 1** 로 간주해 동일하게 비교한다.
- 집계 단위는 instance 가 아니라 **series** (같은 series 안에서 하나의 instance 라도
  태그/값을 갖거나 표준을 지키면 해당 series 는 '있음/준수' 로 간주).

**Metrics (모두 series 기준)**
- `tag_completeness`   = (태그가 있는 series) / (전체 series)
- `value_completeness` = (값이 있는 series)   / (태그가 있는 series)
- `value_conformance`  = (CS 값이 표준을 지키는 series) / (CS 값이 있는 series)  ← `VR == 'CS'` 만


## INPUT: `df_dataset` (real-world DICOM metadata)
| column | DICOM | 설명 |
|---|---|---|
| `IOD` | (0008,0016) SOP Class UID | IOD (참고용, 매칭엔 미사용) |
| `study_instance_uid` | (0020,000D) | Study Instance UID |
| `series_instance_uid` | (0020,000E) | Series Instance UID — **집계 기준 키** |
| `Manufacturer` | (0008,0070) | group_cols 옵션 |
| `ScannerModel` | (0008,1090) | group_cols 옵션 |
| `Tag` | | DICOM Tag |
| `AttributeName` | | Attribute Name |
| `Value` | | Tag value |


In [ ]:
import os
import pandas as pd
import numpy as np
import sys

sys.path.append('Evaluator')
from CxrEssentialTagEvaluator import CxrEssentialTagEvaluator

In [ ]:
data_dir = ''      # df_dataset.parquet 위치
output_dir = ''    # 결과 저장 위치

## Reference: essential tags + CS 허용값

`files/CxrEssentialTags/CxrEssentialTags_ReferenceSet.xlsx` 는
`cxr-metadata-field_260709.xlsx` 의 `CS_allowable_values` 로부터 생성된다
(`DicomStandardRetrieval/build_cxr_essential_reference.py`).
CS 태그의 표준용어(허용값)를 담고 있어 conformance 평가에 사용된다.

In [ ]:
df_standard = pd.read_excel(
    r'../files/CxrEssentialTags/CxrEssentialTags_ReferenceSet.xlsx')
print(df_standard.shape, '| CS tags:', (df_standard['VR'] == 'CS').sum())
df_standard[['Tag', 'Attribute Name', 'VR', 'CS_allowable_values']].head(10)

## Load metadata

In [ ]:
df_dataset = pd.read_parquet(os.path.join(data_dir, '.parquet'))
df_dataset.head()

## Evaluate

In [ ]:
evaluator = CxrEssentialTagEvaluator(df_dataset, df_standard)

### 1) 전체 데이터셋 (group_cols=None)
essential tag 별 series-level tag/value completeness, value conformance 한 줄씩.

In [ ]:
overall = evaluator.analyze(group_cols=None)
overall[['Tag', 'Attribute Name', 'VR', 'total_series', 'series_with_tag',
         'series_with_value', 'tag_completeness', 'value_completeness',
         'value_conformance']].head(30)

In [ ]:
overall.to_excel(os.path.join(output_dir, 'cxr_overall_rates.xlsx'), index=False)

### 2) 그룹별 + 이질성 통계
`group_cols` 로 Manufacturer / ScannerModel 등을 지정하면 그룹별 metric 과
그룹 간 Mean/Std/CV(%)/Range 통계를 얻는다.

In [ ]:
rates, stats = evaluator.analyze_with_stats(group_cols=['Manufacturer'])
stats.head(20)

### 3) Conformance sub-report (CS 태그)

conformance 평가 후, 정확히 일치하지 않는 CS 값을 두 갈래로 분리한다.
**대-소문자는 pass/fail 과 무관** (`CHEST` == `chest` → PASS).

- **partial** : 정확히 일치하진 않지만 허용값을 단어로 **포함** (예: `CHEST` 여야 하는데 `port chest`)
- **none**    : 허용값을 **전혀 포함하지 않음**

각각 건수(count) 와 value_counts 를, `none` 은 비율(%)도 함께 제공한다.

In [ ]:
report = evaluator.conformance_subreport(group_cols=None)

# 태그별 pass/partial/none 건수 및 비율
report['summary']

In [ ]:
# partial: 표준용어를 포함하지만 정확히 일치하지 않는 값의 value_counts
report['partial'].drop(columns=['Defined Values']).head(30)

In [ ]:
# none: 표준용어를 전혀 포함하지 않는 값의 value_counts (+ pct)
report['none'].drop(columns=['Defined Values']).head(30)

In [ ]:
print('partial 건수:', int(report['summary']['n_partial'].sum()))
print('none    건수:', int(report['summary']['n_none'].sum()))

In [ ]:
# CSV 저장: *_summary.csv / *_partial.csv / *_none.csv
evaluator.export_conformance_subreport(
    os.path.join(output_dir, 'cxr_conformance'), group_cols=None)

### 4) (기존 요구) 미준수 unique values CSV
partial + none 을 합쳐 `Tag, Attribute Name, Defined Values, Unconformed Value`
컬럼으로 정리.

In [ ]:
evaluator.export_unconformed_values(
    os.path.join(output_dir, 'cxr_unconformed_values.csv'))